In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit as SS
import tensorflow.python.keras as keras
from matplotlib import pyplot as plt
import os
import tensorflow as tf
import joblib
from sklearn import svm

In [ ]:
# Load data
raw_data = pd.read_excel("36RNA_data.xlsx").to_numpy()

# Shuffle RNA feature order
random_index = [i for i in range(2, 38)]
np.random.shuffle(random_index)
random_data = raw_data[:, random_index]
print('Shuffled feature indices: {}'.format(random_index))

# Prepare input features
X = random_data.reshape((random_data.shape[0], 36))

# Encode binary labels (1: disease, 0: healthy)
y_dict = {'O': 1, 'H': 1, 'L': 1, 'N': 0}
y = np.array([y_dict[i] if i in y_dict else i for i in raw_data[:, 1]])

# Check class distribution
np.unique(y, return_counts=True)

In [ ]:
# Data splitting and model training

# Stratified train-test split
split = SS(n_splits=1, train_size=0.8)
for train_index, test_index in split.split(X, y):
    print("TRAIN:", len(train_index), "TEST:", len(test_index))
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

# Build and train a linear SVM
from sklearn import svm
from sklearn.metrics import classification_report, confusion_matrix

model = svm.LinearSVC(fit_intercept=False, max_iter=10000)
model.fit(X_train, y_train)

In [ ]:
# Model performance evaluation

# Plot the confusion matrix
y_pred = model.predict(X_test)
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
plt.matshow(confusion_matrix(y_test,y_pred),cmap=plt.cm.Reds)

# Plot the ROC curve
from sklearn.metrics import roc_curve, auc
fpr,tpr,threshold = roc_curve(y_test, y_pred)
roc_auc = auc(fpr,tpr)
plt.figure()
plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, color='darkorange', label='ROC curve (area = %0.6f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.01])
plt.ylim([0.0, 1.01])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Test Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

# Plot predicted label distribution
list1, list2 = y_test, y_pred
plt.scatter(range(len(list2)), list2, c=list1)
plt.title("Scatter Plot")
plt.xlabel("Index")
plt.ylabel("Values")
cbar = plt.colorbar()
cbar.set_ticks([0, 1])
plt.show()

# Display model parameters
print(model.coef_)
print(model.intercept_)

In [ ]:
from sklearn.decomposition import PCA
X_2 = PCA(2).fit_transform(X_test)
y_2 = y_test
plt.scatter(X_2[:,0],X_2[:,1],c=y_2)

In [ ]:
#Export model predictions
header_list = list(pd.read_excel("36RNA_data.xlsx")[:0])
header = [header_list[i] for i in random_index]

for label,X_t,y_t in [['train',X_train,y_train],['test',X_test,y_test]]:
    data_file = open('svm_b0_pred_{}.txt'.format(label),'w')
    y_p = model.predict(X_t)
    data_file.writelines('{:12}{:12}'.format('true_0','pred_0'))
    for head in header:
        data_file.writelines('{:15}'.format(head))
    data_file.writelines('\n')
    for i in range(len(y_p)):
        data_file.writelines('{:<12}{:<12.5f}'.format(y_t[i],y_p[i]))
        for j in range(len(X_t[i])):
            data_file.writelines('{:<15.8f}'.format(X_t[i][j]))
        data_file.writelines('\n')
    data_file.close()

In [ ]:
# save
joblib.dump(model, "svm_model.h5")